# Image perceptual example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))


import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.06
n_iter = 300

In [ ]:
global_seed = 2  # Can be None
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

In [ ]:
# Import all available loss classes
from losses.image_image import SSIMLoss, LPIPSLoss, ImageImageCLIPLoss, VGGStyleTransferLoss

In [ ]:
# ============================================================
# LOSS CONFIGURATIONS - Add/remove loss configurations as needed
# ============================================================
loss_configs = [
    ("SSIM", SSIMLoss, {}),
    ("LPIPS", LPIPSLoss, {'backbone': 'vgg'}),
    ("VGGStyle", VGGStyleTransferLoss, {}),
    ("VGG19Style", VGGStyleTransferLoss, {'backbone': 'vgg19'}),
    ("ImageImageCLIP_Base", ImageImageCLIPLoss, {}),
    ("ImageImageCLIP_FineTuned", ImageImageCLIPLoss, {}),
]

# ============================================================
# REFERENCE IMAGE PATHS - Add/remove reference images as needed
# ============================================================

# TODO: Place reference image paths here
reference_image_paths = [
    "",
]

use_fine_tuned_model = True
fine_tuned_model_name = "siglip_blend-training-data_64-output-dim.pt"

In [ ]:
# ============================================================
# SCENE SELECTIONS - Add/remove scenes as needed
# ============================================================
from examples.example_scenes import (
    SpringScene, SciFiRobotScene, BlenderManScene, CarScene, RedCarScene,
    CandleScene, HouseScene, DinoScene, FlowerPotScene, CarStudioScene,
    EinarScene, EinarSmallDomeScene, SpringPortraitScene, SpringPortraitSmallDomeScene
)

scenes = [
    # SciFiRobotScene(),
    SpringScene(device=device),
    # BlenderManScene(),
    # CarScene(),
    # RedCarScene(),
    # CandleScene(),
    # HouseScene(),
    # DinoScene(),
    # FlowerPotScene(),
    # CarStudioScene(),
    # EinarScene(),
    # EinarSmallDomeScene(),
    # SpringPortraitScene(),
    # SpringPortraitSmallDomeScene()
]

In [ ]:
# ============================================================
# LOOP THROUGH COMBINATIONS AND RUN TRAINING
# ============================================================
from utils.train import train_with_criterion
from torchvision.transforms.v2 import RandomChoice, RandomResizedCrop

# Store results for each combination
results = []

for loss_name, loss_class, loss_kwargs in loss_configs:
    for reference_image_path in reference_image_paths:
        for scene in scenes:
            print("\n" + "="*60)
            print(f"Running: {loss_name} | {scene.name} | {os.path.basename(reference_image_path)}")
            print("="*60)
            
            width, height = scene.get_image_resolution()
            
            # Load CLIP model if needed
            clip_model = None
            clip_preprocess = None
            
            if loss_class == ImageImageCLIPLoss:
                
                if not use_fine_tuned_model:  # Empty string = use base model
                    import open_clip
                    clip_model_name = 'ViT-B-16-SigLIP-512'
                    clip_pretrained = 'webli'
                    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
                        clip_model_name, pretrained=clip_pretrained, device=device
                    )
                    clip_model.eval()
                    print(f"Loaded base CLIP model: {clip_model_name}")
                else:
                    # Load fine-tuned model
                    from utils.model.model_utils import create_clip_model_and_tokenizer
                    clip_model, tokenizer, clip_preprocess = create_clip_model_and_tokenizer(model_name = 'ViT-B-16-SigLIP-512', device=device, fine_tune=fine_tuned_model_name)
                    clip_model.eval()
            
            # Create loss criterion
            if loss_class == ImageImageCLIPLoss:
                criterion = loss_class(
                    reference_image_path, clip_model, clip_preprocess,
                    comparison_height=height, comparison_width=width,
                    device=device, **loss_kwargs
                )
            else:
                criterion = loss_class(
                    reference_image_path,
                    comparison_height=height, comparison_width=width,
                    device=device, **loss_kwargs
                )
            
            # Get augmentation for CLIP-based losses
            augmentation = None
            if loss_class == ImageImageCLIPLoss and clip_model is not None:
                size = clip_model.visual.preprocess_cfg['size'] or (224, 224)
                augmentation = RandomChoice([RandomResizedCrop(size=size, scale=(0.4, 1.0), antialias=True)])
            
            # Derive run_name_suffix from reference image path
            run_name_suffix = None
            try:
                prompt_info = criterion.get_prompt_info() or {}
                ref_path = prompt_info.get("reference_image_path")
                if isinstance(ref_path, str) and ref_path.strip():
                    run_name_suffix = os.path.splitext(os.path.basename(ref_path.strip()))[0]
            except Exception:
                run_name_suffix = os.path.splitext(os.path.basename(reference_image_path))[0]
            
            image_name_no_ext = os.path.splitext(os.path.basename(reference_image_path))[0]
            title_prefix = f"ImageImage {loss_name} - {image_name_no_ext}"
            
            print(f"Scene: {scene.name}")
            print(f"Loss: {loss_name}")
            print(f"Reference image: {os.path.basename(reference_image_path)}")
            print(f"Augmentation: {'Yes' if augmentation else 'No'}")
            
            # Run training
            result = train_with_criterion(
                scene,
                lr, n_iter, criterion,
                starting_multiplier_std=(0.4, 0.4, 0.4),
                output_subdirectory_name="image_perceptual_example",
                n_results=4,
                torch_precision=torch_precision,
                augmentation=RandomChoice([RandomResizedCrop(size=(512, 512), scale=(0.4, 1.0), antialias=True)]),
                render_color_space_converter=color_space_converter,
                title_prefix=title_prefix,
                device=device,
                save_every=40,
                model_name=loss_name,
                run_name_suffix=run_name_suffix or image_name_no_ext,
                pretrained_source=fine_tuned_model_name if (loss_class == ImageImageCLIPLoss and use_fine_tuned_model) else "",
                seed=global_seed,
                show_images=True,
            )
            
            results.append({
                'loss': loss_name,
                'scene': scene.name,
                'reference_image': os.path.basename(reference_image_path),
                'result': result
            })

print("\n" + "="*60)
print("ALL TRAINING RUNS COMPLETED")
print("="*60)
for r in results:
    print(f"{r['loss']} | {r['scene']} | {r['reference_image']}")